# 03 - Evaluate `score_reading()` on the test period
Uses the real inference function (so what you measure is what the backend gets), and the simulator's `sim_label` only to MARK the answers.
**Restart the kernel** before running if you re-trained, because the inference module loads the model once at import.

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
DATA_CANDIDATES = [Path("../data/sim_30d.csv"), Path("../data/synthetic_30d.csv")]
DATA = next((p for p in DATA_CANDIDATES if p.exists()), None)
print("using:", DATA)

from energyguard_ml.training import load_csv, time_split
from energyguard_ml.evaluation import run_scoring, summarize, type_report
QUICK_METERS = None    # set to 1 for a fast check (scoring costs ~7 ms per reading)

In [ ]:
df = load_csv(DATA)
_, test = time_split(df, 0.7)
if QUICK_METERS:
    keep = sorted(test.meter_id.unique())[:QUICK_METERS]
    test = test[test.meter_id.isin(keep)]
print("scoring", len(test), "readings ...")
res = run_scoring(test, "../models/model.pkl")

In [ ]:
summary = summarize(res)
for k, v in summary.items():
    print(f"{k:34s} {v:.4f}" if isinstance(v, float) else f"{k:34s} {v}")

How to read it:
- **episode_recall**: share of fault episodes with at least one flagged reading (what the dashboard cares about).
- **false_alarm_rows_per_meter_day**: keep this low; the backend merges repeats, but noisy models waste triage time.
- **latency_ms_p95** must stay under 50 ms (contract).

In [ ]:
report, cm = type_report(res)
print(report)
cm

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
res[res.true_type == "none"].score.plot(kind="hist", bins=50, ax=ax[0], title="score: normal rows", logy=True)
res[res.true_type != "none"].score.plot(kind="hist", bins=50, ax=ax[1], title="score: fault rows")
for a in ax:
    a.axvline(0.5, color="red", ls="--")
plt.tight_layout(); plt.show()
print(res[res.is_anomaly].severity.value_counts())

## Which faults were missed, and which normal rows raised false alarms? (use this to tune)

In [ ]:
missed = res[(res.true_type != "none") & (~res.is_anomaly)]
false_alarms = res[(res.true_type == "none") & (res.is_anomaly)]
print("missed fault rows:", len(missed)); print(missed.groupby("true_type").size())
print("\nfalse alarm rows:", len(false_alarms))
false_alarms.head(10)

## Tuning
If there are too many false alarms, raise `THRESH_PCT` in notebook 02 (for example 99.95). If faults are missed, lower it (99.5) or adjust the rule margins in `training.fit_artifact`.
Re-run 02, restart the kernel, then re-run 03.